# Deep Past Initiative - Akkadian → English Machine Translation

Fine-tune `google/flan-t5-base` on Old Assyrian transliterations to produce English translations.

**Pipeline**: Preprocess → Train → Predict → Submit

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu accelerate sentencepiece

In [ ]:
import csv
import os
import random
import re

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# ── Config ──
MODEL_NAME = "google/flan-t5-base"
PREFIX = "translate Akkadian to English: "
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256
SEED = 42

# Auto-detect Kaggle vs local
if os.path.exists("/kaggle/input"):
    DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
    OUTPUT_DIR = "/kaggle/working/model_output"
    BEST_MODEL_DIR = "/kaggle/working/best_model"
    if not os.path.exists(f"{DATA_DIR}/train.csv"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "train.csv" in files:
                DATA_DIR = root
                break
else:
    DATA_DIR = "data"
    OUTPUT_DIR = "model_output"
    BEST_MODEL_DIR = "best_model"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}, GPUs: {torch.cuda.device_count()}")
print(f"Data dir: {DATA_DIR}")
print(f"Model: {MODEL_NAME}")

## 1. Preprocessing

In [ ]:
def clean_transliteration(text: str) -> str:
    """Clean Akkadian transliteration following competition guidelines."""
    if not text or not isinstance(text, str):
        return ""
    t = text

    # Ḫ/ḫ → H/h  (test data uses H/h only)
    t = t.replace("Ḫ", "H").replace("ḫ", "h")

    # Remove half brackets ˹ ˺
    t = t.replace("˹", "").replace("˺", "")

    # Placeholders to protect gap tokens
    GAP, BIGGAP = "___GAP___", "___BIGGAP___"

    t = re.sub(r"\[x\]", GAP, t)
    t = re.sub(r"\[\s*…\s*…?\s*\]", BIGGAP, t)
    t = re.sub(r"\[\s*\.\.\.\s*\.?\.?\.?\s*\]", BIGGAP, t)

    # Remove remaining [] but keep content
    t = re.sub(r"\[([^\]]*)\]", r"\1", t)

    # Remove <<>> (errant signs, remove content too)
    t = re.sub(r"<<[^>]*>>", "", t)
    # Remove <> (scribal insertions, keep inner text)
    t = re.sub(r"<([^>]*)>", r"\1", t)

    # Standalone ellipsis → big_gap
    t = re.sub(r"…", f" {BIGGAP} ", t)

    # Restore gap tokens
    t = t.replace(GAP, " <gap> ").replace(BIGGAP, " <big_gap> ")

    # Remove scribal notations
    t = re.sub(r"!", "", t)
    t = re.sub(r"\?", "", t)
    t = re.sub(r"/", " ", t)
    t = re.sub(r"\s*:\s*", " ", t)

    # Normalize subscript / superscript numbers
    t = t.translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉ₓ", "0123456789x"))
    t = t.translate(str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789"))

    # Collapse repeated gaps
    t = re.sub(r"(<big_gap>\s*)+", "<big_gap> ", t)
    t = re.sub(r"(<gap>\s*)+", "<gap> ", t)

    return re.sub(r"\s+", " ", t).strip()


def clean_translation(text: str) -> str:
    """Clean English translation text."""
    if not text or not isinstance(text, str):
        return ""
    t = text.strip().strip('"').strip("'")
    t = t.replace('""', '"')
    t = re.sub(r'"\s*$', "", t)
    return re.sub(r"\s+", " ", t).strip()


# Load & preprocess
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

train_df["source"] = train_df["transliteration"].apply(clean_transliteration)
train_df["target"] = train_df["translation"].apply(clean_translation)
train_df = train_df[train_df["source"].str.len() > 0].reset_index(drop=True)

test_df["source"] = test_df["transliteration"].apply(clean_transliteration)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"\nSample source: {train_df['source'].iloc[0][:150]}")
print(f"Sample target: {train_df['target'].iloc[0][:150]}")

## 2. Train/Val Split & Dataset

In [ ]:
# 90/10 split
indices = list(range(len(train_df)))
random.shuffle(indices)
val_size = int(len(train_df) * 0.1)

val_idx = set(indices[:val_size])
train_split = train_df.loc[~train_df.index.isin(val_idx)][["source", "target"]].reset_index(drop=True)
val_split = train_df.loc[train_df.index.isin(val_idx)][["source", "target"]].reset_index(drop=True)

print(f"Train: {len(train_split)}, Val: {len(val_split)}")

train_ds = Dataset.from_pandas(train_split)
val_ds = Dataset.from_pandas(val_split)

## 3. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


def tokenize_fn(examples):
    inputs = [PREFIX + s for s in examples["source"]]
    targets = examples["target"]
    model_inputs = tokenizer(
        inputs, max_length=MAX_SOURCE_LEN, truncation=True, padding=False
    )
    labels = tokenizer(
        text_target=targets, max_length=MAX_TARGET_LEN, truncation=True, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)
print(f"Tokenized train: {len(train_tokenized)}, val: {len(val_tokenized)}")

## 4. Training

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# Train with loss-only eval (no generation during training to save VRAM)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    weight_decay=0.01,
    warmup_steps=50,
    fp16=True,
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=2,
    predict_with_generate=False,  # no generation during eval → saves VRAM
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting training (eval by loss only, generation at the end)...")
trainer.train()
print("Training complete!")

In [ ]:
# Save best model
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"Best model saved to {BEST_MODEL_DIR}")

## 5. Predict & Generate submission.csv

In [ ]:
# Load best model for inference
model.eval()

results = []
for _, row in test_df.iterrows():
    input_text = PREFIX + row["source"]
    inputs = tokenizer(
        input_text, max_length=MAX_SOURCE_LEN, truncation=True, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    results.append({"id": int(row["id"]), "translation": translation})
    print(f"\n--- Test ID {row['id']} ---")
    print(f"Source: {row['source'][:120]}...")
    print(f"Translation: {translation}")

# Write submission
sub_df = pd.DataFrame(results)
sub_df.to_csv("submission.csv", index=False)
print("\nsubmission.csv written!")
sub_df

In [ ]:
# Sanity check: sample validation predictions
print("=== Validation Sample Check ===")
for i in range(min(5, len(val_split))):
    row = val_split.iloc[i]
    input_text = PREFIX + row["source"]
    inputs = tokenizer(
        input_text, max_length=MAX_SOURCE_LEN, truncation=True, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n[{i}] Source : {row['source'][:100]}...")
    print(f"    Predict: {pred[:200]}")
    print(f"    Actual : {row['target'][:200]}")